# Generate Paper Embeddings with SPECTER2

This notebook creates [SPECTER2](https://github.com/allenai/SPECTER2) paper embeddings from a CSV file. It is intended primarily for Google Colab users, where it can run with a free or paid GPU runtime. It can also run locally for users who have a powerful computer with an NVIDIA GPU.

**Author:** Juan Pablo Bascur

**Input:** a CSV file with these columns: `id`, `title`, `abstract`.

**Output:** a CSV file with no header row. The first column is the paper id, followed by 768 embedding values.

## Google Colab GPU Workflow

1. Open the notebook in Google Colab.
2. In the Colab menu, choose **Runtime > Change runtime type**.
3. Set **Hardware accelerator** to **T4 GPU** or another available GPU.
4. Click **Save**.
5. Run the code cell below.
6. Upload your input CSV when prompted. When the embeddings are finished, Colab will download `embeddings.csv` automatically.
7. Check the printed device line. If it says `Using GPU`, Colab is using the GPU runtime, which is usually much faster than CPU for this task. If it says `Using CPU`, repeat steps 2-4 and make sure a GPU is selected.

## Local GPU Workflow

Use this option if you have a powerful local computer with an NVIDIA GPU and a Jupyter notebook interface installed. Examples include JupyterLab, Jupyter Notebook, VS Code notebooks, or the notebook tools included with Anaconda.

1. In a terminal, install the required packages in the Python environment used by this notebook:

   ```bash
   pip install transformers adapters torch pandas numpy
   ```

   If you use Anaconda or Miniconda, run this in Anaconda Prompt or a terminal where conda is available:

   ```bash
   conda install -c conda-forge transformers adapters pytorch pandas numpy
   ```

2. Put your input CSV in the same directory as this notebook and name it `papers.csv`.
3. Run the code cell.
4. Check the printed device line. If it says `Using GPU`, the notebook is using your NVIDIA GPU, which is usually much faster than CPU for this task. If it says `Using CPU`, it will still run, but it may be much slower.
5. When the notebook finishes, `embeddings.csv` will appear in the same directory as this notebook.

Additional configuration: if you want to use a different input filename, output filename, or batch size, change `INPUT_FILE`, `OUTPUT_FILE`, or `BATCH_SIZE` at the top of the code cell.

## Notes

- The first run may take a few minutes because the model is downloaded from Hugging Face.
- If you run out of GPU memory, reduce `BATCH_SIZE` to `32`, `16`, or `8`.
- If your CSV uses different column names, rename them before running this notebook.
- In Colab, the notebook installs `transformers` and `adapters` automatically.


In [ ]:
# Configuration: change these values for local runs.
INPUT_FILE = 'papers.csv'
OUTPUT_FILE = 'embeddings.csv'
BATCH_SIZE = 64
MAX_TOKENS = 512

# Model constants: do not change these.
MODEL_NAME = 'allenai/specter2_base'
ADAPTER_NAME = 'allenai/specter2'
ADAPTER_ALIAS = 'proximity'

# Environment setup
import io
import subprocess
from pathlib import Path

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    subprocess.run(['pip', 'install', '-q', 'transformers', 'adapters'], check=True)

# Imports
import numpy as np
import pandas as pd
import torch
from adapters import AutoAdapterModel
from transformers import AutoTokenizer

# Device check
if torch.cuda.is_available():
    device = 'cuda'
    print(f'Using GPU: {torch.cuda.get_device_name(0)}')
else:
    device = 'cpu'
    print('Using CPU. This will work, but it may be slow for large CSV files.')

# Load input CSV
if IN_COLAB:
    from google.colab import files
    uploaded = files.upload()
    if not uploaded:
        raise ValueError('No file was uploaded.')
    filename = next(iter(uploaded))
    print(f'Uploaded file: {filename}')
    df = pd.read_csv(io.BytesIO(uploaded[filename]))
    output_path = Path(OUTPUT_FILE)
else:
    input_path = Path(INPUT_FILE)
    output_path = Path(OUTPUT_FILE)
    if not input_path.exists():
        raise FileNotFoundError(f'Input file not found: {input_path.resolve()}')
    print(f'Reading input file: {input_path.resolve()}')
    df = pd.read_csv(input_path)

# Validate CSV shape
required_columns = ['id', 'title', 'abstract']
duplicate_columns = df.columns[df.columns.duplicated()].tolist()
if duplicate_columns:
    raise ValueError(f'Duplicate column names found: {duplicate_columns}')

missing_columns = [column for column in required_columns if column not in df.columns]
if missing_columns:
    raise ValueError(f'Missing required columns: {missing_columns}')

if df.empty:
    raise ValueError('The input CSV has no rows.')

print(f'Loaded {len(df)} papers.')

# Load model
print('Loading SPECTER2 model. The first run may take a few minutes...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoAdapterModel.from_pretrained(MODEL_NAME)
model.load_adapter(ADAPTER_NAME, source='hf', load_as=ADAPTER_ALIAS, set_active=True)
model.to(device)
model.eval()
embedding_size = model.config.hidden_size
print(f'Model ready. Embedding size: {embedding_size}')

# Encode papers
n = len(df)
embeddings = np.zeros((n, embedding_size), dtype=np.float32)

for start in range(0, n, BATCH_SIZE):
    end = min(start + BATCH_SIZE, n)
    batch = df.iloc[start:end]
    texts = (
        batch['title'].fillna('').astype(str)
        + ' [SEP] '
        + batch['abstract'].fillna('').astype(str)
    ).tolist()

    inputs = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=MAX_TOKENS,
        return_tensors='pt',
    )
    inputs = {key: value.to(device) for key, value in inputs.items()}

    with torch.no_grad():
        output = model(**inputs)

    embeddings[start:end] = output.last_hidden_state[:, 0, :].detach().cpu().numpy()
    print(f'Encoded {end} / {n}')

# Save output CSV
output_path.parent.mkdir(parents=True, exist_ok=True)
ids = df['id'].astype(str).reset_index(drop=True)
embedding_columns = pd.DataFrame(embeddings)
output_df = pd.concat([ids, embedding_columns], axis=1)
output_df.to_csv(output_path, index=False, header=False, float_format='%.8f')

print(f'Saved embeddings to: {output_path.resolve()}')

if IN_COLAB:
    from google.colab import files
    files.download(str(output_path))